# Prime-acquisition evidence review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** A4 — acquirer concentration among defense primes; F1 — M&A exit  
rate ([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_prime_acquisitions.py`,  
`scripts/data/nano_prime_edgar_filings.py`
**Data as of:** the EDGAR scan snapshot recorded by the generating scripts  

Companion view for firm-level acquisition evidence review: confidence tiers, temporal
qualification, and deal-term outliers. Exploratory-tier and non-citable; counts quoted
anywhere come from the scripts, not from this notebook.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** area keyword-cohort Phase II awards whose firm matches a curated
  prime-acquirer SEC filer; `prime_deal_terms.csv` covers only the manually curated
  `TARGETS` list inside `nano_prime_edgar_filings.py`.
- **Grain:** `prime_acquisitions.csv` is award grain (a firm's awards repeat);
  `prime_deal_terms.csv` is deal grain. Deduplicate to firm grain before counting firms.
- **Keys:** `(award_id, company, award_year)` on the award file; `(firm, acquirer)` on
  deal terms.
- **Caveats:** `ma_date` is the latest SEC *mention* date — an upper bound on when the
  acquisition became visible, not the closing date. `temporal_ok` is necessary, not
  sufficient, for a post-Phase-II acquisition. Detection is a *lower-bound proxy*: only
  acquirers on the curated filer registry are visible.

In [ ]:
ARTIFACTS = {
    "prime acquisitions": REPORT_DIR / "prime_acquisitions.csv",
    "deal terms": REPORT_DIR / "prime_deal_terms.csv",
}
GENERATORS = {
    "prime acquisitions": "scripts/data/nano_prime_acquisitions.py",
    "deal terms": "scripts/data/nano_prime_edgar_filings.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Firm-level evidence bundles

Collapse award-grain rows to one row per (firm, acquiring prime) with the evidence
fields a reviewer needs side by side.

In [ ]:
acquisitions = load_artifact("prime acquisitions")
if acquisitions.empty:
    firm_bundles = pd.DataFrame()
else:
    firm_bundles = (
        acquisitions.groupby(["company", "acquiring_prime", "prime_category"], dropna=False)
        .agg(
            awards=("award_id", "size"),
            confidence=("confidence", "first"),
            signal_source=("signal_source", "first"),
            ma_date=("ma_date", "max"),
            temporal_ok=("temporal_ok", lambda s: sorted(set(s.astype(str)))),
            ma_mention_types=("ma_mention_types", "first"),
        )
        .reset_index()
        .sort_values(["prime_category", "acquiring_prime", "company"])
    )
firm_bundles

## Confidence tiers and temporal qualification

Confidence and `temporal_ok` are separate axes: a high-confidence match can still fail
the post-Phase-II timing test. Keep them crossed, never summed.

In [ ]:
if acquisitions.empty:
    confidence_cross = pd.DataFrame()
else:
    firms = acquisitions.drop_duplicates(subset=["company", "acquiring_prime"])
    confidence_cross = pd.crosstab(
        firms["confidence"], firms["temporal_ok"].astype(str), margins=True
    )
confidence_cross

## Deal-term review and outlier diagnostics

Join extracted deal terms to the acquisition evidence. Outliers (very large or very
small multiples) usually indicate an extraction problem or a deal that bundles more
than the SBIR firm — inspect the filing text before believing the number.

In [ ]:
deal_terms = load_artifact("deal terms")
if deal_terms.empty:
    deal_review = pd.DataFrame()
else:
    deal_review = deal_terms.copy()
    numeric_columns = [
        column for column in deal_review.columns
        if deal_review[column].dtype.kind in "if" and column != "year"
    ]
    if numeric_columns:
        described = deal_review[numeric_columns].describe().T
        print("Numeric deal-term distribution (inspect tails by hand):")
        display(described)
deal_review

## Review log

| Firm / deal | Evidence inspected | Confidence + temporal verdict | Extraction issue? | Follow-up |
|---|---|---|---|---|
| _Draft_ | _Filing/exhibit_ | _Keep axes separate_ | _Yes/no_ | _Rescope TARGETS or registry_ |

Adding an acquirer or rescoping a deal happens in the scripts' curated registries; the
notebook only reviews their output.